<a href="https://colab.research.google.com/github/davidrpugh/data-analytics-with-polars/blob/main/ch11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 11: Filtering and Sorting Rows

This notebook accompanies Chapter 11 of *Python Polars: The Definitive Guide* by Jeroen Janssens and Thijs Nieuwdorp. It is adapted for **Polars 2.1** and **Google Colab**.

Chapter 10 was about columns. This chapter is about **rows**, and two operations in particular:

- **Filtering** keeps a *subset* of the rows, based on their values. It answers questions like *"which tools are cordless?"* or *"which are made by Makita?"*
- **Sorting** *reorders* the rows (the number of rows stays the same). It answers questions like *"which tool is the cheapest?"*

By the end of this notebook you will be able to:

- filter rows with **expressions**, **column names**, and **constraints**, and choose between them
- predict what happens to **missing values** when you filter (rows silently disappear!)
- sort by one column, several columns, or an **expression**, in either direction, with control over `null`s and ties
- sort **nested** data (lists and structs), and know when to avoid it
- use related row operations: dropping missing rows, **slicing**, `top_k`/`bottom_k`, **sampling**, and **semi/anti joins**

Our data is a small table of power tools you'd find in a woodworker's garage.

**How to use this notebook:** run the cells from top to bottom. Some cells *intentionally* trigger errors. Random operations use a **seed**, so the output is reproducible.

## 0. Setup

We pin Polars to the **2.1 series** so everyone sees the same behavior, and use `%pip` so the package is installed into the running notebook kernel.

In [ ]:
%pip install -q "polars==2.1.*"

### Imports and helpers

The assertion turns a stale installation into a clear error message.

- `try_it` runs code and prints its result *or* the error it raises, so demonstrations of errors never stop the notebook. If no error occurs, it says so.
- `download` fetches a data file from the book's repository **only if it isn't already there**. It writes to a temporary name and renames when complete, and raises on HTTP errors instead of saving an error page as if it were data.

In [ ]:
from pathlib import Path
import urllib.request

import polars as pl
from polars.testing import assert_frame_equal


BASE_URL = "https://github.com/davidrpugh/data-analytics-with-polars/raw/refs/heads/main/data"
DATA_DIR = Path("sample_data")


def download(relative_path: str) -> Path:
    """Download BASE_URL/relative_path into DATA_DIR (unless it exists) and return the local path."""
    path = DATA_DIR / relative_path
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary = path.with_name(path.name + ".tmp")
        urllib.request.urlretrieve(f"{BASE_URL}/{relative_path}", temporary)
        temporary.rename(path)
    return path


In [ ]:
print(f"Polars {pl.__version__}")

### The data

Each row is one tool, with its type, product code, brand, whether it's cordless, price, and rotations per minute (`rpm`). Notice that the **nail gun has no `rpm`**: it's missing (`null`). That one missing value will matter several times in this notebook.

In [ ]:
TOOLS_CSV = download("tools.csv")

tools = pl.read_csv(TOOLS_CSV)
tools

## Filtering Rows

`filter` keeps the rows for which a condition is **`True`**. Rows where the condition is `False` *or missing* are removed. (That last part is easy to overlook; we'll come back to it.)

You can describe the condition in three ways: with **expressions** (the most flexible), with **column names** (for Boolean columns), or with **constraints** (keyword arguments).

### Filtering based on expressions

An expression is the most flexible, because you can use any comparison (`==`, `>`, ...) and combine them with Boolean logic (`&` for AND, `|` for OR, `~` for NOT; see Chapter 9). The expression just has to produce a **Boolean** Series. Our favorite tools are the cordless ones made by Makita:

In [ ]:
tools.filter(pl.col("cordless") & (pl.col("brand") == "Makita"))

Note that `pl.col("cordless")` needs no `== True`, because that column is already Boolean. Remember to wrap every comparison in **parentheses** when you combine it with `&` or `|`: those operators bind more tightly than `==` and `>`.

**Commas instead of ampersands.** If your condition is several parts joined with AND, you can pass the parts as **separate arguments**. Polars combines them with AND for you. This reads well for long filters, but it doesn't work for OR (use `|`):

In [ ]:
by_ampersand = tools.filter(pl.col("cordless") & (pl.col("brand") == "Makita"))
by_commas = tools.filter(pl.col("cordless"), pl.col("brand") == "Makita")

In [ ]:
by_commas

In [ ]:
assert_frame_equal(by_ampersand, by_commas)

Expressions can do much more than equality. A few filters you will use constantly:

- `is_in([...])`: the value is one of several
- `is_between(low, high)`: a range (**inclusive** at both ends by default, see Chapter 9)
- the `str` namespace: `str.contains`, `str.starts_with`, ...
- `|`: either condition

In [ ]:
tools.filter(pl.col("brand").is_in(["Bosch", "DeWalt"]))

In [ ]:
tools.filter(pl.col("price").is_between(150, 250))

In [ ]:
tools.filter((pl.col("brand") == "Bosch") | (pl.col("price") < 150))

**A text search pitfall.** `str.contains` is **case sensitive**, and its argument is a regular expression by default. Searching for `"Saw"` finds "Table Saw" and "Miter Saw", but **misses "Jigsaw"**, because that name has a lowercase "saw". To ignore case, lowercase the column first, or start the pattern with `(?i)`:

In [ ]:
print("Case sensitive:  ", tools.filter(pl.col("tool").str.contains("Saw"))["tool"].to_list())
print("Case insensitive:", tools.filter(pl.col("tool").str.contains("(?i)saw"))["tool"].to_list())

### Filtering based on column names

If a column is already Boolean, you can pass its **name** as a string. This is just shorthand for `pl.col(name)`:

In [ ]:
tools.filter("cordless")

You can name several Boolean columns. They must **all** be Boolean, because Polars is strict about this. In Python, many things are "truthy" or "falsy" (an empty string, zero, an empty list count as false), and you might expect Polars to accept them. It doesn't: only Boolean columns and expressions that produce Booleans can be used for filtering. Let's see what Polars says about a text column and a numeric one:

In [ ]:
tools.filter("brand")

In [ ]:
tools.filter(pl.col("price"))

To turn something into a Boolean, use a comparison: `pl.col("brand") != ""` for non-empty text, `pl.col("price") > 0` for positive numbers.


### Filtering based on constraints

The third way is to pass **keyword arguments**: a column name and the value it must equal.

In [ ]:
by_constraints = tools.filter(cordless=True, brand="Makita")

In [ ]:
by_constraints

In [ ]:
assert_frame_equal(by_constraints, by_ampersand)

It's compact, but it has **limitations**:

- only **equality** (`=`) is supported. There's no `>`, `!=`, `is_in`, and so on
- the column name must be a valid Python identifier (no spaces, can't start with a digit, can't be a reserved word like `class`)
- constraints must come **last** if you combine them with other kinds of arguments
- the value must always be given, even for Booleans (`cordless=True`)

Because of these limits we recommend **expressions** for filtering: a little longer, but you never have to switch styles when the condition grows. For instance, this cannot be expressed with constraints:

In [ ]:
tools.filter(pl.col("cordless"), pl.col("brand") == "Makita", pl.col("price") > 200)

### What happens to missing values

A comparison with a missing value gives **missing**, not `True` or `False`, and `filter` keeps only rows that are `True`. So **rows with missing values silently drop out**, whichever way you phrase the condition. The nail gun's `rpm` is missing, so it is in neither of these results:

In [ ]:
fast = tools.filter(pl.col("rpm") > 5000).select("tool", "rpm")
not_fast = tools.filter(~(pl.col("rpm") > 5000)).select("tool", "rpm")


In [ ]:
fast

In [ ]:
not_fast

The two results are not complementary: one row is in *neither*. If "unknown" should count as "not fast", say so explicitly:

In [ ]:
not_fast_or_unknown = tools.filter(~(pl.col("rpm") > 5000) | pl.col("rpm").is_null()).select("tool", "rpm")

In [ ]:
not_fast_or_unknown

**Filters on lazy data.** In Chapter 5 you saw that a LazyFrame can push a filter down to the *scan*, so rows that don't match are never loaded. Read the plan from the bottom up. The filter and the column selection should appear in the scan line:

In [ ]:
print(pl.scan_csv(TOOLS_CSV).filter(pl.col("cordless")).select("tool", "price").explain())

## Sorting Rows

Sorting **reorders** the rows according to the values in one or more columns. The number of rows doesn't change. Typical questions: *Which brand has the most tools? What is the cheapest tool?*

You can sort numbers, strings (alphabetically, see below), dates and times (chronologically), and even nested types.

### Sorting based on a single column

The default order is **ascending** (smallest first):

In [ ]:
tools.sort("price")

### Sorting in reverse

Set `descending=True`. Pandas users: the argument isn't called `ascending`, and Polars lets you know:

In [ ]:
tools.sort("price", descending=True)

In [ ]:
tools.sort("price", ascending=False)

### Sorting based on multiple columns

Pass several column names. The first is the main key, and later ones break **ties**. Here we sort by brand, and within each brand by price:

In [ ]:
tools.sort("brand", "price")

With several columns, `descending` can be a **list of Booleans**, one per column. Here: brands A to Z, but the most expensive tool first within each brand. The list must have exactly as many entries as there are sort columns, or Polars raises an error:

In [ ]:
tools.sort("brand", "price", descending=[False, True])

In [ ]:
tools.sort("brand", "price", descending=[False, True, True])

### Missing values in a sort

Where do `null`s go? By default, **first**. (Pandas puts them last.) Use `nulls_last=True` to move them to the end. Look at the nail gun, whose `rpm` is missing:

In [ ]:
tools.sort("rpm")["tool"].to_list()[:3]

In [ ]:
tools.sort("rpm", nulls_last=True)["tool"].to_list()[-3:]

### Sorting based on expressions

You can sort by a *computed* value that is **not** a column of the DataFrame. Here: rotations per minute per dollar. A temporary Series is created for the sort and then thrown away, so it doesn't show up in the result.

Because the nail gun's `rpm` is missing, its ratio is missing too, so it comes first (see the previous section).

In [ ]:
tools.sort(pl.col("rpm") / pl.col("price"))

**Strings sort in byte order, not dictionary order.** Capital letters come *before* all lowercase letters, so `"Cherry"` sorts before `"apple"`. For a case-insensitive sort, sort by an expression that lowercases the text, with the original as a tie-breaker:

In [ ]:
names = pl.DataFrame({"name": ["banana", "Cherry", "apple", "Banana"]})

print("default:         ", names.sort("name")["name"].to_list())
print("case-insensitive:", names.sort(pl.col("name").str.to_lowercase(), "name")["name"].to_list())

### Ties and stability

When several rows have the **same** value, what order do they come out in? By default Polars doesn't promise to keep their original order, because sorting runs on several threads. If you need ties to stay in their input order (a **stable** sort), pass `maintain_order=True`, which costs a little performance.

In [ ]:
stable = tools.with_row_index().sort("brand", maintain_order=True)

stable

### Sorting nested data types

Lists and structs can be sorted too. **Lists** compare *element by element*: first elements first, then second elements for ties, and so on. A shorter list that is a prefix of a longer one comes first:

In [ ]:
list_df = pl.DataFrame({"lists": [[2, 2], [2, 1, 3], [1]]})

list_df.sort("lists")

**Structs** compare *field by field*, in the order of the fields:

In [ ]:
struct_df = pl.DataFrame(
    {
        "structs": [
            {"a": 1, "b": 2, "c": 3},
            {"a": 1, "b": 3, "c": 1},
            {"a": 1, "b": 1, "c": 2},
        ]
    }
)

struct_df.sort("structs")

Often what you want to sort by is some **property** of a nested value, not the nested value itself. For practice, build a table with one row per brand, where each row holds *all of that brand's tools* in a list of structs. (Group order isn't guaranteed, so we sort by brand to keep the table stable.)

In [ ]:
tools_collection = (
    tools.group_by("brand")
    .agg(collection=pl.struct(pl.all()))
    .sort("brand")
)

tools_collection

Sort the brands by **how many tools they have**, using the list's length as the key:

In [ ]:
tools_collection.sort(pl.col("collection").list.len(), descending=True)

Or by the **average price** of their tools. This needs to reach *inside* the list: `list.eval` runs an expression on the elements, `pl.element()` refers to each element, and `struct.field` picks the `price` out of each struct:

In [ ]:
mean_price_key = pl.col("collection").list.eval(pl.element().struct.field("price")).list.mean()

tools_collection.sort(mean_price_key)

**Materialize first, sort second.** When a sort key gets complicated, first create it as a real column with `with_columns`, check that the numbers look right, and only then turn it into a sort again:

In [ ]:
(
    tools_collection.with_columns(mean_price=mean_price_key)
                    .select("brand", "mean_price")
                    .sort("mean_price")
)

**A simpler way: don't nest in the first place.** Sorting nested values is a good exercise, but the nesting was only needed to *create* the exercise. If all you want is brands ordered by their average price, aggregate directly, with no lists or structs. We check that it gives the same order:

In [ ]:
brand_summary = (
    tools.group_by("brand")
    .agg(n_tools=pl.len(), mean_price=pl.col("price").mean())
    .sort("mean_price")
)

brand_summary

## Related Row Operations

### Filtering missing values

If your analysis can't cope with missing values, `drop_nulls` keeps only the rows **without** any. By default it checks **every** column. Pass column names to check only those, which is usually what you intend:

In [ ]:
print("rows in total:                    ", tools.height)
print("rows without a missing rpm:       ", tools.drop_nulls("rpm").height)
print("rows without any missing value:   ", tools.drop_nulls().height)

`drop_nulls()` is shorthand for a filter on all columns.

### Slicing

Sometimes you want rows by their **position**, regardless of their values:

- `head(n)` and `tail(n)`: the first or last `n` rows
- `slice(offset, length)`: `length` rows, starting at position `offset`. Positions start at **0**, and a negative offset counts from the end
- `gather_every(n)`: every `n`th row

Careful with `slice`. The second argument is a **length**, not an end position. The book's text says `slice(2, 7)` keeps "the third to the seventh row", but that call keeps **seven** rows, starting at the third. We add a row index so the positions are visible:

In [ ]:
indexed = tools.with_row_index().select("index", "tool")

print("slice(2, 5): positions", indexed.slice(2, 5)["index"].to_list())
print("slice(2, 7): positions", indexed.slice(2, 7)["index"].to_list())
print("slice(-3):   positions", indexed.slice(-3)["index"].to_list())

> **A word of caution.** Selecting rows by position is usually a sign that you know something about the data that Polars doesn't (such as "the table is sorted"), so it can't use that knowledge, and the result changes if the order changes (for example, after a parallel operation). Whenever you can, express the condition on the **values** instead (`filter`), or **sort first** and then slice, so the order is explicit.


### Top and bottom

`top_k(k, by=...)` keeps the `k` rows with the **largest** values, and `bottom_k` the **smallest**. This is typically faster than sorting everything, but the rows come back in **no guaranteed order**, so sort afterwards if order matters. We check that `top_k` plus a sort agrees with a full sort plus `head`:

In [ ]:
top_three = tools.top_k(3, by="price").sort("price", descending=True)

top_three

In [ ]:
tools.bottom_k(3, by="price").sort("price")

### Sampling

`sample` keeps **random** rows: a fixed number (`n`) or a fraction of the rows (`fraction`). Without a **seed**, you get a different answer every time you run it, so for reproducible analyses always pass `seed=`. Run each cell below twice to see the difference. Also, `with_replacement=True` allows a row to be picked more than once.

In [ ]:
tools.sample(fraction=0.2, seed=42).select("tool", "price")

In [ ]:
print("seeded, 1st:", tools.sample(n=3, seed=1729)["tool"].to_list())
print("seeded, 2nd:", tools.sample(n=3, seed=1729)["tool"].to_list())
print("unseeded:   ", tools.sample(n=3)["tool"].to_list())

`sample` exists only on a DataFrame, because it needs to know how many rows there are before choosing. A LazyFrame doesn't know yet. Let's see what your version says:

In [ ]:
tools.lazy().sample(n=3, seed=1).collect()

### Semi-joins and anti-joins

A third way to filter rows is to **join** with another DataFrame:

- a **semi-join** keeps the rows of the left table that **have** a match in the right table (and adds no columns from the right)
- an **anti-join** keeps the rows that have **no** match

This is the natural tool when the "condition" is *membership in another table*. Our second table lists saws, including some we don't own (bandsaw, chainsaw, seesaw).

In [ ]:
saws = pl.DataFrame(
    {"tool": ["Table Saw", "Plunge Cut Saw", "Miter Saw", "Jigsaw", "Bandsaw", "Chainsaw", "Seesaw"]}
)

tools.join(saws, how="semi", on="tool")

In [ ]:
tools.join(saws, how="anti", on="tool")

For a short list of values, `is_in` does the same job without a second DataFrame. Check that both ways give the same answer, and that the semi-join and anti-join together account for every row:

In [ ]:
semi = tools.join(saws, how="semi", on="tool")
anti = tools.join(saws, how="anti", on="tool")
by_is_in = tools.filter(pl.col("tool").is_in(saws["tool"].to_list()))


In [ ]:
semi

In [ ]:
anti

In [ ]:
by_is_in

Use `is_in` for a short, fixed list; use a semi-join or anti-join when the other table is large or comes from elsewhere (Chapter 14 covers joins).



## Takeaways

- `filter` keeps rows where the condition is **True**. Rows where it is **False or missing** are removed.
- Describe the condition with **expressions** (most flexible), **Boolean column names**, or **constraints** (equality only). We recommend expressions.
- Filtering is strict: the condition must be **Boolean**, and **parenthesize each comparison** before combining with `&` or `|`.
- Comparisons with `null` give `null`, so rows with missing values vanish from *both* a filter and its negation. Handle `null` explicitly when it matters.
- `sort` orders ascending by default; use `descending=True` (not `ascending=`). `nulls_last=True` moves missing values to the end; `maintain_order=True` makes ties keep their input order.
- You can sort by **several columns**, by a **list of directions**, and by an **expression** that isn't a column.
- Strings sort in **byte order** (capitals first). For case-insensitive sorting, sort by a lowercased expression.
- Nested values sort element-wise (lists) or field-wise (structs), but it's simpler not to nest: aggregate directly.
- `drop_nulls`, `head`, `tail`, `slice`, `top_k`, `bottom_k`, `sample`, and semi/anti-joins are all ways of keeping a subset of rows. `slice`'s second argument is a **length**. `top_k` is **unordered**. Always **seed** `sample`.
- Selecting by **position** is fragile. Prefer conditions on **values**, or sort first.

### Next up

In Chapter 12 we work with textual, temporal, and nested data types.

### Try it yourself

1. Filter `tools` to cordless tools that cost **more than the average price of all tools**. (Hint: an aggregation like `pl.col("price").mean()` is an expression and can appear inside a comparison.)

2. Find the three most expensive tools **per brand**. (Hint: sort by price, then use `group_by("brand").head(3)`, or look into window functions with `over`.)

3. Sort the tools by brand (A to Z) and then by `rpm` (highest first), with the tool that has no `rpm` at the **end** of its brand.

4. Use an anti-join to find the tools whose **brand** isn't in a list of brands you give.

5. Compare `tools.sample(n=3)` before and after you set `seed=` and re-run each cell several times. What stays the same?